# 第十三课｜仿真不是芯片

前面我们已经会写 RTL、跑 testbench、看 waveform。仿真可以回答“在这个模型里，输入和时钟这样变化时，逻辑行为对不对”。

今天只解决一个问题：

> **仿真通过以后，为什么还不能说电路已经能在 FPGA 上工作？**

主要新概念：**从 RTL 到可配置芯片还要经过综合、实现与时序检查。**

## 1. 概念账本

**已经知道：** RTL、clock、register、testbench、waveform、simulation。

**今天学习：**
- **综合（synthesis）**：把 RTL 转换成可由 FPGA 逻辑资源实现的网络；
- **实现（implementation）**：把综合后的逻辑映射、放置并连接到具体 FPGA 资源；
- **时序分析（timing analysis）**：检查信号传播能否满足 clock period 等时间约束；
- **配置比特流（bitstream）**：用于配置 FPGA 可编程资源的数据。

**只预告：** 开发板、host 通信、DDR、AXI 会在后面分课学习。

## 2. simulation 与 synthesis 回答不同问题

simulation 关心**行为**：状态更新、threshold、reset、spike 等规则是否符合 test oracle。

synthesis 关心**可实现结构**：这些 RTL 能否变成查找表、register、片上存储和互连。

因此，“testbench 全绿”和“电路可以在目标频率下运行”是两个需要分别验证的结论。

## 3. 从 RTL 到 bitstream

```mermaid
flowchart LR
  RTL["RTL / SystemVerilog"] --> SIM["simulation"]
  RTL --> SYN["synthesis"]
  SYN --> IMP["implementation"]
  IMP --> TIM["timing analysis"]
  TIM --> BIT["bitstream"]
  BIT --> FPGA["configured FPGA"]
```

这里先建立流程概念，不要求记住某家厂商工具的按钮或菜单。

## 4. timing 为什么会失败？

一个 register 的输出经过组合逻辑，到达下一个 register 的输入需要时间。目标 clock period 给这条路径设了一个截止时间。

本课使用一个简化模型：

- **critical path**：最长的组合路径延迟；
- **slack** = clock period − critical path delay；
- slack ≥ 0：这个教学模型满足目标时序；
- slack < 0：目标 clock 太快。

真实 timing analysis 还会考虑 setup/hold、clock uncertainty 等更多约束；本课暂不展开。

## 5. Run：算一个最小 timing budget

先预测：下面哪条是 critical path？5 ns 的 clock period 能否容纳它？

In [ ]:
path_delays_ns = [2.2, 4.8, 3.1]
clock_period_ns = 5.0

critical_path_ns = max(path_delays_ns)
slack_ns = clock_period_ns - critical_path_ns
meets_timing = slack_ns >= 0

print("critical path:", critical_path_ns, "ns")
print("clock period:", clock_period_ns, "ns")
print("slack:", slack_ns, "ns")
print("meets timing:", meets_timing)


## 6. Observe

最长路径是 4.8 ns，目标周期是 5.0 ns，因此 slack 为 +0.2 ns。

这只说明**当前教学 timing model**没有超出目标周期，不等于已经完成真实器件的全部 timing sign-off。

## 7. Try It

把 `clock_period_ns` 改成 4.0。先预测：

1. critical path 会不会改变？
2. slack 的正负号会怎样改变？
3. RTL 的逻辑功能有没有因此改变？

这个实验用于区分“功能行为”和“能否在规定时间内完成”。

## 8. 作业

打开：

[第 13 课作业：读懂一个 timing budget](../../exercises/zh/13_simulation_is_not_chip.ipynb)

作业要求从一组路径延迟计算 critical path、slack 和是否满足时序。

## 9. AI Task

把一份 synthesis/timing 摘要交给 AI，让它分别解释 resource usage、critical path 和 slack。检查它有没有把 simulation pass 错说成 timing pass。

## 10. Human Check

不用 AI，你应该能解释 simulation、synthesis、implementation、timing analysis 分别在问什么；为什么 waveform 正确仍可能 timing fail；bitstream 与 RTL 源码是什么关系；为什么本课不需要购买 FPGA 板。

## 11. Engineering Handoff

本课对应 `RMD-011A`：先做 FPGA toolchain dry run，不买板也可以完成。

## 12. 项目追踪 Project Trace

- Lesson: `LSN-013`
- Mapping: `RMD-011A`
- Evidence: synthesis report + timing summary
- Boundary: no physical board required

## 13. Exit Ticket

你能解释“仿真正确”和“芯片能按目标时钟运行”为什么必须分别验证。